# 5. Positional embedding and self-attention

## Importing Libraries

The required libraries are imported from Keras. The imported layers are used to build a Transformer-style architecture, including attention, normalization, and dense layers.

In [ ]:
import keras
from keras.datasets import imdb
from keras.layers import Input, Dense, GlobalAveragePooling1D, Dropout, MultiHeadAttention, LayerNormalization, Layer, Embedding
from keras.models import Model

## Loading and Preparing the Data

The IMDB dataset is loaded using Keras. The data is already converted into numbers, so each word is represented by an integer.

All sequences are made the same length by applying padding. Each review is limited to 250 words, and only the 10000 most common words are kept.

In [ ]:
max_features = 10000
max_len = 250

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = keras.utils.pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = keras.utils.pad_sequences(x_test, maxlen=max_len, padding='post')

## Token and Positional Embedding

A custom embedding layer is defined in this cell.

The layer combines two types of embeddings:
- Token embeddings (representing the meaning of words)
- Positional embeddings (representing the position of words in a sentence)

These are added together so that both the content and the order of words can be understood by the model.

In [ ]:
class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)
        self.pos_emb = Embedding(input_dim=seq_len, output_dim=emb_dim)

    def call(self, x):
        seq_len = keras.ops.shape(x)[-1]
        positions = keras.ops.arange(start=0, stop=seq_len, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

## Setting Hyperparameters

The main hyperparameters of the model are defined.
These include:
- the embedding dimension (size of word vectors)
- the number of attention heads
- the size of the feedforward network

These values control the complexity of the model.

In [ ]:
embed_dim = 32
num_heads = 2
key_dim = embed_dim // num_heads
ff_dim = 128  # Feedforward network size

## Building the Model

The model is built using a Transformer-style encoder block. The model includes:
- multi-head attention
- residual connections
- layer normalization
- a feedforward network

In [ ]:
inputs = Input(shape=(max_len,))

# Embedding
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)

attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)

# Residual connection 1
residual_1 = x + attention_output

# Layer Normalization 1
normalization_1 = LayerNormalization()(residual_1)

# Feedforward network
ff_network = Dense(ff_dim, activation="relu")(normalization_1)
ff_network = Dense(embed_dim)(ff_network)

# Residual connection 2
residual_2 = normalization_1 + ff_network

# Layer Normalization 2
normalization_2 = LayerNormalization()(residual_2)

x = GlobalAveragePooling1D()(normalization_2)
x = Dropout(0.5)(x)
outputs = Dense(1, activation="sigmoid")(x)

## Compiling the Model

The model is compiled using **binary crossentropy loss**, the **Adam optimizer**, and **accuracy** as the evaluation metric.

In [ ]:
model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer_2[0]… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 250, 32)   │          0 │ token_and_positi… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 250, 128)  │      4,224 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 250, 32)   │      4,128 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 250, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         33 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 340,737 (1.30 MB)

 Trainable params: 340,737 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

## Training the Model

The model is trained using the training data. A validation split is used to monitor performance during training.

In [11]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 85ms/step - accuracy: 0.7714 - loss: 0.4621 - val_accuracy: 0.8682 - val_loss: 0.3292
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.9102 - loss: 0.2341 - val_accuracy: 0.8904 - val_loss: 0.2705
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.9391 - loss: 0.1684 - val_accuracy: 0.8814 - val_loss: 0.2947
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 98ms/step - accuracy: 0.9590 - loss: 0.1166 - val_accuracy: 0.8690 - val_loss: 0.3621
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 60s 96ms/step - accuracy: 0.9736 - loss: 0.0826 - val_accuracy: 0.8646 - val_loss: 0.5203


## Evaluating the Model

The model is evaluated on the test dataset to measure its performance.

In [12]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy = {test_acc:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 59ms/step - accuracy: 0.8521 - loss: 0.5603
Test accuracy = 0.8521
